# Tenuto — Expressive Score-to-Performance AI Engine

Predicts human performance nuance (rubato, micro-timing, velocity, articulation, sustain pedal) from sheet music or MIDI scores.

### Stage 1: Setup Repository & Environment
**Note:** To enable GPU acceleration in Colab, go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# 1. Clone repo if needed or pull latest changes
import os
if not os.path.exists("/content/tenuto"):
    get_ipython().system("git clone https://github.com/kyleconciso/tenuto.git /content/tenuto")
else:
    get_ipython().system("cd /content/tenuto && git pull")

# 2. Set working directory & PYTHONPATH globally
%cd /content/tenuto
%env PYTHONPATH=/content/tenuto:.

# 3. Verify PyTorch GPU & install dependencies
import torch
print("Current Directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Switch to GPU in Colab via: Runtime > Change runtime type > T4 GPU")

%pip install -q partitura mido scipy tqdm matplotlib huggingface_hub midi2audio pandas pyarrow rclone


### Stage 2: Connect Google Drive & Sync Preprocessed Dataset
Mount your Google Drive. Upload your local  (or ) to . It will automatically extract into  and skip re-extraction if already present.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# 2. Idempotent check for preprocessed dataset
import os, zipfile, glob
target_processed = "/content/tenuto/data/processed"
os.makedirs(target_processed, exist_ok=True)

pt_files = glob.glob(os.path.join(target_processed, "**/*.pt"), recursive=True)
if len(pt_files) >= 1000:
    print(f"[TenutoColab] ✅ Preprocessed dataset already extracted ({len(pt_files)} files found). Skipping extraction!")
else:
    gdrive_zips = [
        "/content/drive/MyDrive/Tenuto/processed.zip",
        "/content/drive/MyDrive/Tenuto/storage.zip",
        "/content/drive/MyDrive/processed.zip"
    ]
    zip_path = None
    for zp in gdrive_zips:
        if os.path.exists(zp):
            zip_path = zp
            break
            
    if zip_path:
        print(f"[TenutoColab] Extracting preprocessed zip from Google Drive: '{zip_path}'...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(target_processed)
        print("[TenutoColab] 🎉 Extraction complete!")
    else:
        print("[TenutoColab] No preprocessed zip found in Google Drive ('/content/drive/MyDrive/Tenuto/processed.zip').")


### Stage 3: Preprocess Dataset (Skipped automatically if preprocessed zip exists)

In [ ]:
%cd /content/tenuto
import glob
pt_files = glob.glob("/content/tenuto/data/processed/**/*.pt", recursive=True)
if len(pt_files) >= 1000:
    print(f"[TenutoColab] ✅ Preprocessed dataset ready ({len(pt_files)} files). Skipping preprocessing step!")
else:
    get_ipython().system("PYTHONPATH=/content/tenuto python3 -m src.download_dataset --dataset combined --pianocore_subset PianoCoRe-A*")
    get_ipython().system("PYTHONPATH=/content/tenuto python3 -m src.preprocess --data_dir ./data --processed_dir ./data/processed --max_samples 10000")


### Stage 4: Train Transformer Backbone (Auto-Resumes & Auto-Saves to Google Drive)

In [ ]:
%cd /content/tenuto
# Auto-resumes and auto-saves checkpoints per epoch to /content/drive/MyDrive/Tenuto/checkpoints/
get_ipython().system("PYTHONPATH=/content/tenuto python3 -m src.train --model_type transformer --data_dir ./data/processed --in_features 40 --epochs 20 --batch_size 16 --lr 0.0001")


### Stage 5: Expressive Inference

In [ ]:
%cd /content/tenuto
get_ipython().system("PYTHONPATH=/content/tenuto python3 -m src.infer --score data/asap/Balakirev/Islamey/xml_score.musicxml --checkpoint checkpoints/best_transformer_model.pth --model_type transformer --output_midi output_expressive.mid")


### Stage 6: Listenable Audio Comparison 🎧

In [ ]:
%cd /content/tenuto
get_ipython().system("apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm timidity")

import sys
sys.path.insert(0, "/content/tenuto")
from src.audio import play_audio_in_colab

print("🎵 1. Playing Original Flat Score (Mechanical):")
play_audio_in_colab("data/asap/Balakirev/Islamey/midi_score.mid", title="Original Flat Score")

print("
🎵 2. Playing Original Human Performance (Ground Truth):")
play_audio_in_colab("data/asap/Balakirev/Islamey/CHEN04.mid", title="Human Performance")

print("
🎵 3. Playing Tenuto AI Generated Performance:")
play_audio_in_colab("output_expressive.mid", title="Tenuto AI Expressive Performance")
